In [1]:
import heapq
import math

class TrangThaiGhepHinh:
    def __init__(self, bang, cha=None, buoc_di="", g=0):
        self.bang = bang
        self.cha = cha
        self.buoc_di = buoc_di
        self.g = g
        self.h = self.tinh_khoang_cach_manhattan()
        self.f = self.g + self.h
    def __lt__(self, other):
        return self.f < other.f

    def __eq__(self, other):
        return self.bang == other.bang

    def __hash__(self):
        return hash(self.bang)

    def tinh_khoang_cach_manhattan(self):
        khoang_cach = 0
        kich_thuoc = len(self.bang)
        for hang in range(kich_thuoc):
            for cot in range(kich_thuoc):
                o_vuong = self.bang[hang][cot]
                if o_vuong == 0:
                    continue

                hang_muc_tieu, cot_muc_tieu = divmod(o_vuong - 1, kich_thuoc) if o_vuong != 0 else (kich_thuoc - 1, kich_thuoc - 1)
                khoang_cach += abs(hang - hang_muc_tieu) + abs(cot - cot_muc_tieu)
        return khoang_cach

    def lay_vi_tri_o_trong(self):
        kich_thuoc = len(self.bang)
        for hang in range(kich_thuoc):
            for cot in range(kich_thuoc):
                if self.bang[hang][cot] == 0:
                    return hang, cot
        return -1, -1

    def lay_cac_trang_thai_lan_can(self):
        cac_trang_thai_lan_can = []
        hang, cot = self.lay_vi_tri_o_trong()
        kich_thuoc = len(self.bang)

        cac_buoc_di = {
            "LÊN": (-1, 0),
            "XUỐNG": (1, 0),
            "TRÁI": (0, -1),
            "PHẢI": (0, 1)
        }

        for ten_buoc_di, (d_hang, d_cot) in cac_buoc_di.items():
            hang_moi, cot_moi = hang + d_hang, cot + d_cot

            if 0 <= hang_moi < kich_thuoc and 0 <= cot_moi < kich_thuoc:
                bang_moi = [list(row) for row in self.bang]
                bang_moi[hang][cot], bang_moi[hang_moi][cot_moi] = bang_moi[hang_moi][cot_moi], bang_moi[hang][cot]
                cac_trang_thai_lan_can.append(TrangThaiGhepHinh(tuple(tuple(row) for row in bang_moi), self, ten_buoc_di, self.g + 1))
        return cac_trang_thai_lan_can

    def hien_thi(self):
        kich_thuoc = len(self.bang)
        for hang in range(kich_thuoc):
            for cot in range(kich_thuoc):
                print(f"{self.bang[hang][cot]:2d}", end=" ")
            print()
        print(f"g={self.g}, h={self.h}, f={self.f}")


def giai_ghep_hinh_astar(bang_ban_dau):
    kich_thuoc = len(bang_ban_dau)
    bang_muc_tieu_list = [[(hang * kich_thuoc + cot + 1) for cot in range(kich_thuoc)] for hang in range(kich_thuoc)]
    bang_muc_tieu_list[kich_thuoc-1][kich_thuoc-1] = 0
    bang_muc_tieu = tuple(tuple(row) for row in bang_muc_tieu_list)

    trang_thai_ban_dau = TrangThaiGhepHinh(tuple(tuple(row) for row in bang_ban_dau))
    tap_mo = [trang_thai_ban_dau]
    heapq.heapify(tap_mo)
    tap_dong = {trang_thai_ban_dau.bang: trang_thai_ban_dau.g}
    while tap_mo:
        trang_thai_hien_tai = heapq.heappop(tap_mo)
        if trang_thai_hien_tai.bang == bang_muc_tieu:

            duong_di = []
            while trang_thai_hien_tai:
                duong_di.append(trang_thai_hien_tai)
                trang_thai_hien_tai = trang_thai_hien_tai.cha
            return duong_di[::-1]

        for trang_thai_lan_can in trang_thai_hien_tai.lay_cac_trang_thai_lan_can():
            if trang_thai_lan_can.bang in tap_dong and trang_thai_lan_can.g >= tap_dong[trang_thai_lan_can.bang]:
                continue
            if trang_thai_lan_can.bang not in tap_dong or trang_thai_lan_can.g < tap_dong[trang_thai_lan_can.bang]:
                tap_dong[trang_thai_lan_can.bang] = trang_thai_lan_can.g
                heapq.heappush(tap_mo, trang_thai_lan_can)

    return None

print("### Ví dụ cho bài toán 3x3 (Kích thước=3) ###")
bang_ban_dau_3x3 = (
    (1, 2, 3),
    (4, 0, 5),
    (6, 7, 8)
)

print("Bảng 3x3 ban đầu:")
for row in bang_ban_dau_3x3:
    print(row)

duong_di_giai_3x3 = giai_ghep_hinh_astar(bang_ban_dau_3x3)

if duong_di_giai_3x3:
    print(f"\nTìm thấy lời giải trong {len(duong_di_giai_3x3) - 1} bước:")
    for i, trang_thai in enumerate(duong_di_giai_3x3):
        print(f"\nBước {i} (Di chuyển: {trang_thai.buoc_di})")
        trang_thai.hien_thi()
else:
    print("\nKhông tìm thấy lời giải cho bài toán 3x3.")
print("\n### Ví dụ cho bài toán 4x4 (Kích thước=4) - Có thể tốn nhiều tính toán ###")
bang_ban_dau_4x4 = (
    (1, 2, 3, 4),
    (5, 6, 7, 8),
    (9, 10, 11, 12),
    (13, 14, 0, 15)
)

print("Bảng 4x4 ban đầu:")
for row in bang_ban_dau_4x4:
    print(row)


### Ví dụ cho bài toán 3x3 (Kích thước=3) ###
Bảng 3x3 ban đầu:
(1, 2, 3)
(4, 0, 5)
(6, 7, 8)

Tìm thấy lời giải trong 14 bước:

Bước 0 (Di chuyển: )
 1  2  3 
 4  0  5 
 6  7  8 
g=0, h=6, f=6

Bước 1 (Di chuyển: PHẢI)
 1  2  3 
 4  5  0 
 6  7  8 
g=1, h=5, f=6

Bước 2 (Di chuyển: XUỐNG)
 1  2  3 
 4  5  8 
 6  7  0 
g=2, h=6, f=8

Bước 3 (Di chuyển: TRÁI)
 1  2  3 
 4  5  8 
 6  0  7 
g=3, h=7, f=10

Bước 4 (Di chuyển: TRÁI)
 1  2  3 
 4  5  8 
 0  6  7 
g=4, h=6, f=10

Bước 5 (Di chuyển: LÊN)
 1  2  3 
 0  5  8 
 4  6  7 
g=5, h=7, f=12

Bước 6 (Di chuyển: PHẢI)
 1  2  3 
 5  0  8 
 4  6  7 
g=6, h=8, f=14

Bước 7 (Di chuyển: XUỐNG)
 1  2  3 
 5  6  8 
 4  0  7 
g=7, h=7, f=14

Bước 8 (Di chuyển: PHẢI)
 1  2  3 
 5  6  8 
 4  7  0 
g=8, h=6, f=14

Bước 9 (Di chuyển: LÊN)
 1  2  3 
 5  6  0 
 4  7  8 
g=9, h=5, f=14

Bước 10 (Di chuyển: TRÁI)
 1  2  3 
 5  0  6 
 4  7  8 
g=10, h=4, f=14

Bước 11 (Di chuyển: TRÁI)
 1  2  3 
 0  5  6 
 4  7  8 
g=11, h=3, f=14

Bước 12 (Di chuyển: XU

In [ ]:
#bai2

class TrangThaiTSP:
    def __init__(self, thanh_pho_hien_tai, mat_na_da_tham, chi_phi_duong_di, cha=None, duong_di=None):
        self.thanh_pho_hien_tai = thanh_pho_hien_tai
        self.mat_na_da_tham = mat_na_da_tham
        self.chi_phi_duong_di = chi_phi_duong_di
        self.cha = cha
        self.duong_di = duong_di if duong_di is not None else [thanh_pho_hien_tai]
        self.h = 0
        self.f = 0

    def __lt__(self, other):
        return self.f < other.f

    def __eq__(self, other):

        return self.thanh_pho_hien_tai == other.thanh_pho_hien_tai and self.mat_na_da_tham == other.mat_na_da_tham

    def __hash__(self):
        return hash((self.thanh_pho_hien_tai, self.mat_na_da_tham))

def tinh_ham_heuristic(thanh_pho_hien_tai, mat_na_da_tham, so_thanh_pho, ma_tran_khoang_cach, thanh_pho_bat_dau):

    thanh_pho_chua_tham = []
    for i in range(so_thanh_pho):
        if not (mat_na_da_tham & (1 << i)):
            thanh_pho_chua_tham.append(i)

    if not thanh_pho_chua_tham:
        return ma_tran_khoang_cach[thanh_pho_hien_tai][thanh_pho_bat_dau]

    min_hien_tai_den_chua_tham = float('inf')
    for next_city in thanh_pho_chua_tham:
        min_hien_tai_den_chua_tham = min(min_hien_tai_den_chua_tham, ma_tran_khoang_cach[thanh_pho_hien_tai][next_city])

    min_chua_tham_den_bat_dau = float('inf')
    for city_u in thanh_pho_chua_tham:
        min_chua_tham_den_bat_dau = min(min_chua_tham_den_bat_dau, ma_tran_khoang_cach[city_u][thanh_pho_bat_dau])

    h_value = min_hien_tai_den_chua_tham + min_chua_tham_den_bat_dau
    return h_value

def giai_tsp_astar(ma_tran_khoang_cach, thanh_pho_bat_dau=0):
    so_thanh_pho = len(ma_tran_khoang_cach)

    mat_na_muc_tieu = (1 << so_thanh_pho) - 1
    mat_na_ban_dau = (1 << thanh_pho_bat_dau)
    trang_thai_ban_dau = TrangThaiTSP(thanh_pho_hien_tai=thanh_pho_bat_dau, mat_na_da_tham=mat_na_ban_dau, chi_phi_duong_di=0)
    trang_thai_ban_dau.h = tinh_ham_heuristic(trang_thai_ban_dau.thanh_pho_hien_tai, trang_thai_ban_dau.mat_na_da_tham, so_thanh_pho, ma_tran_khoang_cach, thanh_pho_bat_dau)
    trang_thai_ban_dau.f = trang_thai_ban_dau.chi_phi_duong_di + trang_thai_ban_dau.h

    tap_mo = [trang_thai_ban_dau]
    heapq.heapify(tap_mo)
    tap_dong = {}
    tap_dong[(trang_thai_ban_dau.thanh_pho_hien_tai, trang_thai_ban_dau.mat_na_da_tham)] = trang_thai_ban_dau.chi_phi_duong_di

    chi_phi_giai_phap_tot_nhat = float('inf')
    duong_di_giai_phap_tot_nhat = None

    while tap_mo:
        trang_thai_hien_tai = heapq.heappop(tap_mo)


        if trang_thai_hien_tai.mat_na_da_tham == mat_na_muc_tieu:

            chi_phi_cuoi_cung = trang_thai_hien_tai.chi_phi_duong_di + ma_tran_khoang_cach[trang_thai_hien_tai.thanh_pho_hien_tai][thanh_pho_bat_dau]
            if chi_phi_cuoi_cung < chi_phi_giai_phap_tot_nhat:
                chi_phi_giai_phap_tot_nhat = chi_phi_cuoi_cung
                duong_di_giai_phap_tot_nhat = trang_thai_hien_tai.duong_di + [thanh_pho_bat_dau]
            continue
        if trang_thai_hien_tai.f >= chi_phi_giai_phap_tot_nhat:
            continue

        for thanh_pho_tiep_theo in range(so_thanh_pho):

            if not (trang_thai_hien_tai.mat_na_da_tham & (1 << thanh_pho_tiep_theo)):
                new_mat_na_da_tham = trang_thai_hien_tai.mat_na_da_tham | (1 << thanh_pho_tiep_theo)
                new_chi_phi_duong_di = trang_thai_hien_tai.chi_phi_duong_di + ma_tran_khoang_cach[trang_thai_hien_tai.thanh_pho_hien_tai][thanh_pho_tiep_theo]


                trang_thai_moi = TrangThaiTSP(
                    thanh_pho_hien_tai=thanh_pho_tiep_theo,
                    mat_na_da_tham=new_mat_na_da_tham,
                    chi_phi_duong_di=new_chi_phi_duong_di,
                    cha=trang_thai_hien_tai,
                    duong_di=trang_thai_hien_tai.duong_di + [thanh_pho_tiep_theo]
                )


                trang_thai_moi.h = tinh_ham_heuristic(trang_thai_moi.thanh_pho_hien_tai, trang_thai_moi.mat_na_da_tham, so_thanh_pho, ma_tran_khoang_cach, thanh_pho_bat_dau)
                trang_thai_moi.f = trang_thai_moi.chi_phi_duong_di + trang_thai_moi.h


                if (trang_thai_moi.thanh_pho_hien_tai, trang_thai_moi.mat_na_da_tham) not in tap_dong or \
                   trang_thai_moi.chi_phi_duong_di < tap_dong[(trang_thai_moi.thanh_pho_hien_tai, trang_thai_moi.mat_na_da_tham)]:
                    tap_dong[(trang_thai_moi.thanh_pho_hien_tai, trang_thai_moi.mat_na_da_tham)] = trang_thai_moi.chi_phi_duong_di
                    heapq.heappush(tap_mo, trang_thai_moi)

    return duong_di_giai_phap_tot_nhat, chi_phi_giai_phap_tot_nhat


ma_tran_khoang_cach_vi_du = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
]

print("### Bài toán Người giao hàng (Traveling Salesperson Problem - TSP) với thuật toán A* ###")
print("Ma trận khoảng cách (ví dụ 4 thành phố):")
for row in ma_tran_khoang_cach_vi_du:
    print(row)

duong_di, chi_phi = giai_tsp_astar(ma_tran_khoang_cach_vi_du, thanh_pho_bat_dau=0)

if duong_di:
    print(f"\nĐường đi ngắn nhất: {duong_di} (bắt đầu và kết thúc tại thành phố {thanh_pho_bat_dau})")
    print(f"Tổng chi phí: {chi_phi}")
else:
    print("\nKhông tìm thấy lời giải.")

print("\n--- Lưu ý về độ phức tạp ---")
print("Thuật toán A* có thể hiệu quả hơn tìm kiếm vét cạn, nhưng vẫn có độ phức tạp cao (NP-hard) cho TSP.")
print("Đối với số lượng thành phố lớn (thường > 15-20), việc giải quyết TSP bằng A* có thể rất tốn kém về thời gian và bộ nhớ.")
print("Hàm heuristic được sử dụng ở đây là đơn giản. Các heuristic mạnh hơn (ví dụ: dựa trên Cây bao trùm tối thiểu - MST) sẽ cần cài đặt phức tạp hơn.")